# Unsupervised Learning: County Profile Clustering

Clustering workflow for non-FMD county profiles, including DBSCAN/K-Means style analysis, cluster visualization, and held-out FMD comparison.

This active notebook was split from the original combined learning notebook so the clean repository has clearly delineated supervised and unsupervised work. The original combined notebook is archived under `archive/notebooks/original_combined/`.


## 0. Setup

**Note:** Run the optional install cell only if your notebook environment is missing packages. Also, the core modeling sections expect `scikit-learn`; `UMAP` and `geopandas` are optional because the notebook includes fallbacks.

In [ ]:
# Optional: uncomment and run if your environment is missing these packages.
# %pip install scikit-learn matplotlib seaborn scipy umap-learn geopandas shapely

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

try:
    import umap.umap_ as umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

try:
    from scipy.stats import f_oneway
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

try:
    import geopandas as gpd
    from shapely import wkt
    HAS_GEO = True
except Exception:
    HAS_GEO = False

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

## 1. Load the Imputed County Panel

The dataset is expected at `data/processed/county_panel_imputed.csv` relative to the clean repository root.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "processed" / "county_panel_imputed.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Run this notebook from the clean repository root.")

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}")
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]:,}")
display(raw.head())

In [ ]:
raw.info()

In [ ]:
summary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "n_unique": raw.nunique(dropna=True),
})
display(summary)

print("Years:", sorted(raw["year"].unique()))
print("Unique counties:", raw["geoid"].nunique())
print("Duplicate county-year rows:", raw.duplicated(["geoid", "year"]).sum())

## 2. Target Engineering

We want to see if current-year air quality and socioeconomic conditions can predict **next-year FMD**. We therefore sort by county and year, shift `mental_health_prevalence` one year forward within each county, and drop rows whose following year is unavailable or non-consecutive. We do this to help avoid leakage.

In [ ]:
df = raw.copy()
df["geoid"] = df["geoid"].astype(str).str.zfill(5)
df = df.sort_values(["geoid", "year"]).reset_index(drop=True)

# Shift target and year within each county.
df["next_year"] = df.groupby("geoid")["year"].shift(-1)
df["next_year_fmd"] = df.groupby("geoid")["mental_health_prevalence"].shift(-1)

# Keep only true one-year-ahead prediction rows.
supervised_df = df.loc[df["next_year"].eq(df["year"] + 1)].copy()
supervised_df["next_year"] = supervised_df["next_year"].astype(int)

print(f"Rows available for one-year-ahead supervised modeling: {len(supervised_df):,}")
print(supervised_df.groupby(["year", "next_year"]).size())
display(supervised_df[["geoid", "county_name", "state_name", "year", "mental_health_prevalence", "next_year", "next_year_fmd"]].head())

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(supervised_df["next_year_fmd"], bins=30, kde=True)
plt.title("Distribution of Next-Year Frequent Mental Distress")
plt.xlabel("Next-year FMD prevalence (%)")
plt.ylabel("County-years")
plt.tight_layout()
plt.show()

## 3. Feature Sets

The supervised models use current-year features only. The unsupervised models exclude FMD and FMD confidence intervals, so that the cluster evaluation can treat mental distress as a held-out outcome.

In [ ]:
TARGET = "next_year_fmd"
CURRENT_FMD = "mental_health_prevalence"
ID_COLS = ["geoid", "county_name", "state_name", "year", "next_year", "geometry"]
FMD_RELATED = ["mental_health_prevalence", "lower_ci", "upper_ci", "next_year_fmd"]

# Convert comma-formatted population field if present.
for frame in (supervised_df, df):
    if "total_population" in frame.columns:
        frame["total_population_numeric"] = (
            frame["total_population"].astype(str).str.replace(",", "", regex=False).replace("nan", np.nan).astype(float)
        )

numeric_cols = supervised_df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    c for c in numeric_cols
    if c not in FMD_RELATED + ["next_year"]
]

# `geoid` became string, so it is not in numeric_cols; keep IDs out if a future read infers otherwise.
feature_cols = [c for c in feature_cols if c not in ID_COLS]

air_quality_features = [c for c in feature_cols if any(token in c.lower() for token in ["aqi", "days", "co", "no2", "ozone", "pm2.5", "pm10", "unhealthy", "hazardous", "good", "moderate"])]
socioeconomic_features = [c for c in feature_cols if c not in air_quality_features]

print(f"Model features ({len(feature_cols)}):")
for col in feature_cols:
    print("-", col)

print("\nAir-quality feature count:", len(air_quality_features))
print("Socioeconomic/other feature count:", len(socioeconomic_features))

In [ ]:
feature_overview = supervised_df[feature_cols + [CURRENT_FMD, TARGET]].describe().T
feature_overview["missing"] = supervised_df[feature_cols + [CURRENT_FMD, TARGET]].isna().sum()
display(feature_overview)

## 5. Unsupervised Learning

### Design

The clustering stage uses only environmental and socioeconomic features, excluding FMD and confidence intervals. FMD is brought back afterward as a hold-out outcome for interpretation and ANOVA.

UMAP is attempted first because it matches the project plan. If `umap-learn` is unavailable, the notebook falls back to PCA so the workflow remains runnable.

In [ ]:
cluster_df = df.copy()
cluster_df = cluster_df[cluster_df["year"] == cluster_df["year"].max()].copy()

# Use latest year for cross-sectional county archetypes. Drop columns that should not define clusters.
cluster_numeric_cols = cluster_df.select_dtypes(include=[np.number]).columns.tolist()
exclude_from_clustering = set(["next_year", "next_year_fmd", "mental_health_prevalence", "lower_ci", "upper_ci"])
cluster_features = [c for c in cluster_numeric_cols if c not in exclude_from_clustering]

# Keep model features if available, plus any numeric socioeconomic additions.
cluster_features = [c for c in cluster_features if c in feature_cols or c == "total_population_numeric"]

print(f"Clustering {len(cluster_df):,} counties from year {cluster_df['year'].iloc[0]}")
print(f"Cluster features ({len(cluster_features)}):")
for col in cluster_features:
    print("-", col)

X_cluster = cluster_df[cluster_features].replace([np.inf, -np.inf], np.nan)
X_cluster = X_cluster.fillna(X_cluster.median(numeric_only=True))
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

In [ ]:
# if HAS_UMAP:
#     reducer = umap.UMAP(
#         n_neighbors=30,
#         min_dist=0.08,
#         n_components=2,
#         metric="euclidean",
#         random_state=RANDOM_STATE,
#     )
#     embedding = reducer.fit_transform(X_cluster_scaled)
#     embedding_method = "UMAP"
# else:
#     reducer = PCA(n_components=2, random_state=RANDOM_STATE)
#     embedding = reducer.fit_transform(X_cluster_scaled)
#     embedding_method = "PCA fallback"
#     print("umap-learn is not installed. Using PCA fallback; install umap-learn for the planned UMAP embedding.")

# cluster_df["embed_1"] = embedding[:, 0]
# cluster_df["embed_2"] = embedding[:, 1]
# print("Embedding method:", embedding_method)

In [ ]:
if HAS_UMAP:
    reducer = umap.UMAP(
        n_neighbors=30,
        min_dist=0.08,
        n_components=2,
        metric="euclidean",
        random_state=RANDOM_STATE,
    )
    embedding = reducer.fit_transform(X_cluster_scaled)
    embedding_method = "UMAP"
else:
    reducer = PCA(n_components=2, random_state=RANDOM_STATE)
    embedding = reducer.fit_transform(X_cluster_scaled)
    embedding_method = "PCA fallback"
    print("umap-learn is not installed. Using PCA fallback; install umap-learn for the planned UMAP embedding.")

cluster_df["embed_1"] = embedding[:, 0]
cluster_df["embed_2"] = embedding[:, 1]

# Alias used by downstream DBSCAN cells
embedding_for_cluster = embedding

print("Embedding method:", embedding_method)
print("Embedding shape:", embedding_for_cluster.shape)

In [ ]:
# Search DBSCAN settings on the 2D embedding. Noise points are labeled -1.
search_rows = []
for eps in np.round(np.linspace(0.15, 1.50, 10), 2):
    for min_samples in [5, 10, 20, 35, 50]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(embedding_for_cluster)
        non_noise = labels != -1
        n_clusters = len(set(labels[non_noise]))
        noise_pct = 1 - non_noise.mean()
        sil_embed = np.nan
        sil_scaled = np.nan
        if n_clusters >= 2 and non_noise.sum() > n_clusters:
            sil_embed = silhouette_score(embedding_for_cluster[non_noise], labels[non_noise])
            sil_scaled = silhouette_score(X_cluster_scaled[non_noise], labels[non_noise])
        search_rows.append({
            "eps": eps,
            "min_samples": min_samples,
            "clusters": n_clusters,
            "noise_pct": noise_pct,
            "silhouette_embedding": sil_embed,
            "silhouette_scaled_features": sil_scaled,
        })

search_df = pd.DataFrame(search_rows)
valid_search = search_df[search_df["clusters"].between(2, 12)].copy()
if valid_search.empty:
    best_params = search_df.sort_values(["clusters", "noise_pct"], ascending=[False, True]).iloc[0]
else:
    best_params = valid_search.sort_values(["silhouette_embedding", "silhouette_scaled_features"], ascending=False).iloc[0]

display(search_df.sort_values(["silhouette_embedding", "silhouette_scaled_features"], ascending=False).head(15))
print("Selected DBSCAN parameters:")
print(best_params)

In [ ]:
dbscan = DBSCAN(eps=float(best_params["eps"]), min_samples=int(best_params["min_samples"]))
cluster_df["cluster"] = dbscan.fit_predict(embedding_for_cluster)

cluster_counts = cluster_df["cluster"].value_counts().sort_index().rename_axis("cluster").reset_index(name="count")
cluster_counts["share"] = cluster_counts["count"] / len(cluster_df)
display(cluster_counts)

non_noise = cluster_df["cluster"] != -1
n_clusters = cluster_df.loc[non_noise, "cluster"].nunique()
if n_clusters >= 2:
    print("Silhouette on embedding:", silhouette_score(embedding_for_cluster[non_noise.to_numpy()], cluster_df.loc[non_noise, "cluster"]))
    print("Silhouette on scaled original features:", silhouette_score(X_cluster_scaled[non_noise.to_numpy()], cluster_df.loc[non_noise, "cluster"]))
else:
    print("DBSCAN found fewer than two non-noise clusters; adjust eps/min_samples above.")

### Cluster Visualization and Hold-Out FMD Test

The scatterplot shows how the clustering behaves in the low-dimensional space. The summaries and ANOVA test ask whether the clusters differ on mental distress despite FMD not being used to form them.

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=cluster_df,
    x="embed_1",
    y="embed_2",
    hue="cluster",
    palette="tab10",
    s=28,
    alpha=0.8,
    linewidth=0,
)
plt.title(f"{embedding_method} Embedding Colored by DBSCAN Cluster")
plt.xlabel(f"{embedding_method} 1")
plt.ylabel(f"{embedding_method} 2")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
cluster_profile = (
    cluster_df.groupby("cluster")
    .agg(
        counties=("geoid", "count"),
        avg_fmd=("mental_health_prevalence", "mean"),
        median_fmd=("mental_health_prevalence", "median"),
        avg_median_aqi=("Median AQI", "mean"),
        avg_max_aqi=("Max AQI", "mean"),
        avg_income=("median_income", "mean"),
        avg_poverty_count=("poverty_count", "mean"),
        avg_unemployed_count=("unemployed_count", "mean"),
        avg_population=("population", "mean"),
    )
    .sort_values("avg_fmd", ascending=False)
)
display(cluster_profile)

In [ ]:
plot_profile = cluster_profile.reset_index().melt(
    id_vars=["cluster"],
    value_vars=["avg_fmd", "avg_median_aqi", "avg_max_aqi", "avg_income", "avg_poverty_count", "avg_unemployed_count"],
    var_name="metric",
    value_name="value",
)

# Standardize metrics for a readable grouped comparison.
plot_profile["z_value"] = plot_profile.groupby("metric")["value"].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) else 0)

plt.figure(figsize=(11, 6))
sns.barplot(data=plot_profile, x="cluster", y="z_value", hue="metric")
plt.axhline(0, color="black", linewidth=1)
plt.title("Cluster Profiles: Standardized Metric Averages")
plt.xlabel("DBSCAN cluster")
plt.ylabel("Standard deviations from metric mean")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
if HAS_SCIPY and cluster_df.loc[cluster_df["cluster"] != -1, "cluster"].nunique() >= 2:
    groups_for_anova = [
        group["mental_health_prevalence"].dropna().values
        for _, group in cluster_df[cluster_df["cluster"] != -1].groupby("cluster")
    ]
    f_stat, p_value = f_oneway(*groups_for_anova)
    print(f"ANOVA on held-out FMD across non-noise clusters: F={f_stat:.3f}, p={p_value:.4g}")
else:
    print("ANOVA skipped: scipy is unavailable or DBSCAN found fewer than two non-noise clusters.")

In [ ]:
# Identify the features that make each cluster stand out relative to the full latest-year county sample.
feature_means = cluster_df.groupby("cluster")[cluster_features].mean()
overall_means = cluster_df[cluster_features].mean()
overall_stds = cluster_df[cluster_features].std(ddof=0).replace(0, np.nan)
z_profiles = (feature_means - overall_means) / overall_stds

top_drivers = []
for cluster_id, row in z_profiles.iterrows():
    strongest = row.abs().sort_values(ascending=False).head(6).index
    for feature in strongest:
        top_drivers.append({
            "cluster": cluster_id,
            "feature": feature,
            "z_difference": row[feature],
            "direction": "above average" if row[feature] > 0 else "below average",
        })

top_drivers_df = pd.DataFrame(top_drivers)
display(top_drivers_df)

## 6. Optional Geographic Views

We wanted to use choropleth-inspired spatial visualization. The imputed CSV contains point WKT geometries, so this section maps county centroids/points rather than full county polygons. (If someone wants to come in and later add a county boundary shapefile, they would need to replace the point geometry with polygon geometries for a true choropleth).

In [ ]:
if HAS_GEO and "geometry" in cluster_df.columns:
    geo_df = cluster_df.copy()
    geo_df["geometry"] = geo_df["geometry"].apply(wkt.loads)
    geo_df = gpd.GeoDataFrame(geo_df, geometry="geometry", crs="EPSG:4326")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    geo_df.plot(column="cluster", categorical=True, legend=True, markersize=6, alpha=0.8, ax=axes[0])
    axes[0].set_title("DBSCAN Cluster by County Point")
    axes[0].set_axis_off()

    geo_df.plot(column="mental_health_prevalence", cmap="magma_r", legend=True, markersize=6, alpha=0.8, ax=axes[1])
    axes[1].set_title("Frequent Mental Distress by County Point")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()
else:
    print("Geographic plot skipped: geopandas/shapely is unavailable or geometry column is missing.")

## 7. Export Model Outputs for Reporting

These CSVs make it easier to create report tables and visuals without rerunning every cell.

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling" / "primary_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

supervised_predictions_path = OUTPUT_DIR / "supervised_holdout_predictions.csv"
model_metrics_path = OUTPUT_DIR / "supervised_model_metrics.csv"
cluster_assignments_path = OUTPUT_DIR / "unsupervised_cluster_assignments.csv"
cluster_profile_path = OUTPUT_DIR / "unsupervised_cluster_profile.csv"

pred_df.to_csv(supervised_predictions_path, index=False)
results_export = pd.concat([
    results_df.assign(section="base_models"),
    tuned_results_df.drop(columns=["best_params"], errors="ignore").rename(columns={"holdout_rmse": "rmse", "holdout_mae": "mae", "holdout_r2": "r2"}).assign(section="tuned_models"),
], ignore_index=True, sort=False)
results_export.to_csv(model_metrics_path, index=False)
cluster_df[["geoid", "county_name", "state_name", "year", "mental_health_prevalence", "cluster", "embed_1", "embed_2"]].to_csv(cluster_assignments_path, index=False)
cluster_profile.to_csv(cluster_profile_path)

print("Saved:")
for path in [supervised_predictions_path, model_metrics_path, cluster_assignments_path, cluster_profile_path]:
    print(path)